<div class="alert alert-info">
    <b>2.4 Filter Database (Ionic Balance Error)</b><br>
    Dieses Notebook filtert die aktuellste 'Komplette_Datenbank' basierend auf dem Ionenbilanzfehler (IBE), wie in der Main-Ions-Six Analyse definiert.<br>
    -> <b>Ziel:</b> Nur Proben behalten, deren IBE innerhalb von +/- 5% liegt UND deren Temperatur >= 10°C ist<br>
    -> <b>Output:</b> Speichert eine NEUE Version (mit Zeitstempel) der 'Komplette_Datenbank.csv' für den nächsten Pipeline-Schritt (3.1 Preprocessing).
</div>

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime
import shutil

In [2]:
# ------------------------- Einstellungen -------------------------
IBE_THRESHOLD = 5.0  # Prozent (+/-)
TEMP_THRESHOLD = 10.0 # Grad Celsius (Minimum)
REQUIRED_IONS = ['Na', 'Ca', 'Mg', 'Cl', 'HCO3', 'SO4']

# ------------------------- Valenz-Wörterbuch zur Normalisierung -------------------------
ION_SPECS = {
    'Na':   {'valence': 1, 'mass': 22.99, 'type': 'cation', 'regex': r'Na_in_([a-zA-Z0-9/]+)'},
    'Ca':   {'valence': 2, 'mass': 40.08, 'type': 'cation', 'regex': r'Ca_in_([a-zA-Z0-9/]+)'},
    'Mg':   {'valence': 2, 'mass': 24.31, 'type': 'cation', 'regex': r'Mg_in_([a-zA-Z0-9/]+)'},
    'Cl':   {'valence': 1, 'mass': 35.45, 'type': 'anion',  'regex': r'Cl_in_([a-zA-Z0-9/]+)'},
    'HCO3': {'valence': 1, 'mass': 61.02, 'type': 'anion',  'regex': r'HCO3_in_([a-zA-Z0-9/]+)'},
    'SO4':  {'valence': 2, 'mass': 96.06, 'type': 'anion',  'regex': r'SO4_in_([a-zA-Z0-9/]+)'},
}

In [3]:

def calculate_ibe(df):
    # ------------------------- Valenzen & Molmassen -------------------------
    # (Wiederholung der Specs aus Dictionary oben, hier zur Verwendung)
    
    # 1. Extrahiere Konzentrationen (regex) und konvertiere zu numerisch
    # wir nutzen das ION_SPECS dictionary von oben
    
    cation_sum_meq = 0.0
    anion_sum_meq = 0.0
    
    valid_cols_mask = pd.Series([True] * len(df), index=df.index)

    for ion, specs in ION_SPECS.items():
        # Finde Spalte, die auf Regex passt
        # Da wir die genauen Namen oft nicht wissen (Einheitsteil), suchen wir:
        matches = [c for c in df.columns if re.search(specs['regex'], c)]
        
        if not matches:
            # Falls Ionen fehlen -> können wir IBE nicht berechnen -> Zeile ungültig
            print(f"Warnung: Ion {ion} nicht in Spalten gefunden. IBE nicht berechenbar.")
            valid_cols_mask[:] = False
            continue
            
        col_name = matches[0] # Nehme die erste passende (sollte eindeutig sein)
        
        # Werte holen (numerisch erzwingen)
        vals = pd.to_numeric(df[col_name], errors='coerce').fillna(0.0)
        
        # Umrechnung in meq/L:
        # mmol/L * Valenz = meq/L
        # mg/L / Molmasse * Valenz = meq/L
        # Hier Annahme: Einheit ist im Spaltennamen kodiert.
        # Aber unsere Regex 'Na_in_([a-zA-Z0-9/]+)' fängt die Einheit.
        
        unit_match = re.search(specs['regex'], col_name)
        unit = unit_match.group(1) if unit_match else "unknown"
        
        # Konvertierung zu mmol/L, falls nötig
        if unit.lower() in ['mg/l', 'mg_l']:
            vals_mmol = vals / specs['mass']
        elif unit.lower() in ['mmol/l', 'mmol_l', 'mm']:
            vals_mmol = vals
        elif unit.lower() in ['umol/l', 'umol_l']:
            vals_mmol = vals / 1000.0
        else:
            # Fallback: Annahme mmol/L bei unbekannter Einheit. 
            # Vereinfachung: Warnung wird hier ausgelassen.
            vals_mmol = vals 

        meq_vals = vals_mmol * specs['valence']
        
        if specs['type'] == 'cation':
            cation_sum_meq += meq_vals
        else:
            anion_sum_meq += meq_vals
            
    # 2. IBE Berechnung
    # IBE = (Sigma Kationen - Sigma Anionen) / (Sigma Kationen + Sigma Anionen) * 100
    
    diff = cation_sum_meq - anion_sum_meq
    total = cation_sum_meq + anion_sum_meq
    
    # Vermeide Division durch Null
    ibe = (diff / total) * 100.0
    
    # Wo total fast 0 ist -> NaN oder 0 setzen
    ibe[total < 1e-9] = np.nan
    
    # Maske für Zeilen, die valide Ionen hatten (nicht alle 0 oder fehlend)
    # Zusätzlich: Mindestens ein Kation und ein Anion sollten > 0 sein für sinnvolle Balance?
    # Hier nehmen wir einfache Berechenbarkeit.
    
    return ibe, valid_cols_mask


In [4]:

base_dir = Path.cwd()

notebooks_root = base_dir.parent.parent
# Input-Pfad: Nach_Acquisition
db_input_path = notebooks_root / "1_Acquisition" / "1.1_Data-Acquisition-Wrapper" / "Angepasste_Datenbanken" / "Nach_Acquisition" / "Komplette_Datenbank"

if not db_input_path.exists():
    raise FileNotFoundError(f"Datenbank-Archivpfad nicht gefunden: {db_input_path}")

# ------------------------- Finde den neuesten Zeitstempel-Ordner -------------------------
all_folders = [f for f in db_input_path.iterdir() if f.is_dir()]
if not all_folders:
     raise FileNotFoundError(f"Keine Zeitstempel-Ordner in {db_input_path} gefunden.")

latest_folder = max(all_folders, key=lambda x: x.name)
input_csv = latest_folder / "Komplette_Datenbank.csv"

print(f"Lade aktuellste Datenbank (Input): {input_csv}")
df = pd.read_csv(input_csv, low_memory=False)
print(f"Anzahl geladener Zeilen: {len(df)}")


Lade aktuellste Datenbank (Input): C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_Acquisition\Komplette_Datenbank\2026-01-08_11-19-58\Komplette_Datenbank.csv


Anzahl geladener Zeilen: 175099


<div class="alert alert-info">
    <b>2. Logik zur IBE-Berechnung</b>
</div>

<div class="alert alert-info">
    <b>3. Filter anwenden</b>
</div>

In [5]:
print("Berechne IBE...")
ibe_values, valid_sum_mask = calculate_ibe(df)

# ------------------------- Wende Schwellenwert-Filter an (IBE und Temperatur) -------------------------
temp_vals = pd.to_numeric(df['temperature_in_c'], errors='coerce')

# Logik: Behalten wenn Temp >= 10 ODER Temp fehlt (NaN)
temp_condition = (temp_vals >= TEMP_THRESHOLD) | temp_vals.isna()

filter_mask = (ibe_values.abs() <= IBE_THRESHOLD) & valid_sum_mask & temp_condition

df_filtered = df[filter_mask].copy()

removed_count = len(df) - len(df_filtered)
print(f"\nFilter Bericht:")
print(f"Zeilen Original:  {len(df)}")
print(f"Zeilen Gefiltert: {len(df_filtered)}")
print(f"Entfernte Zeilen: {removed_count} ({(removed_count/len(df))*100:.2f}%)")

Berechne IBE...

Filter Bericht:
Zeilen Original:  175099
Zeilen Gefiltert: 94264
Entfernte Zeilen: 80835 (46.17%)


<div class="alert alert-info">
    <b>4. Gefilterte Datenbank speichern</b>
</div>

In [6]:

# ------------------------- Output Pfad definieren -------------------------
# Sicherstellen, dass Basispfade definiert sind
if 'base_dir' not in locals():
    base_dir = Path.cwd()
if 'notebooks_root' not in locals():
    notebooks_root = base_dir.parent.parent

db_output_path = notebooks_root / "1_Acquisition" / "1.1_Data-Acquisition-Wrapper" / "Angepasste_Datenbanken" / "Nach_IBF-und-Temp_Filter" / "Komplette_Datenbank"

if not db_output_path.exists():
    db_output_path.mkdir(parents=True, exist_ok=True)
    
print(f"Ausgabepfad gesetzt auf: {db_output_path}")


Ausgabepfad gesetzt auf: C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_IBF-und-Temp_Filter\Komplette_Datenbank


In [7]:
# ------------------------- Erzeuge neuen Zeitstempel-Ordner -------------------------
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
new_output_folder = db_output_path / timestamp

if not new_output_folder.exists():
    new_output_folder.mkdir(parents=True)

output_file = new_output_folder / "Komplette_Datenbank.csv"

import pathlib
_p = pathlib.Path(output_file)
if not _p.parent.exists(): _p.parent.mkdir(parents=True, exist_ok=True)
df_filtered.to_csv(output_file, index=False)
print(f"Gefilterte Datenbank gespeichert unter:\n{output_file}")

# ------------------------- Überprüfung für den nächsten Pipeline-Schritt -------------------------
print(f"Erstellter Ordner: {new_output_folder.name}")

Gefilterte Datenbank gespeichert unter:
C:\Users\lucca\OneDrive\SPEICHER\Hochschule\7. Semester\Abschlussarbeit Bearbeitung\Jupyter Notebooks\1_Acquisition\1.1_Data-Acquisition-Wrapper\Angepasste_Datenbanken\Nach_IBF-und-Temp_Filter\Komplette_Datenbank\2026-01-08_15-14-53\Komplette_Datenbank.csv
Erstellter Ordner: 2026-01-08_15-14-53
